# TorchRL Modules

In this tutorial, we will delve into the core functionality of TorchRL in terms of policy construction. We will primarily focus on how to build policies for different environments, and a little about how to setup the agent to explore and exploit.

* Note: this notebook was modified from the official tutorials provided by torchRL, you can check it out here: https://docs.pytorch.org/rl/stable/tutorials

### Install TorchRL

First let's install torchrl and tensordict.


In [ ]:
#Option 1: easy
#If you are running this from colab you can run:
#!pip install tensordict
#!pip install torchrl

#Option 2: harder
#if you are using your own compputer you should create a virtual environment, activate it, and then use pip to install the needed 
#conda create -n torchrl-env python=3.11 -y
#conda activate torchrl-env
#pip install torchrl tensordict torch torchvision 'gymnasium[all]'
#Note: for GPU support this might be a little different, and might vary by device!

### TensorDictModules

Similar to how environments interact with instances of TensorDict, the modules used to represent policies and value functions also do the same. The core idea is simple: encapsulate a standard Module (or any other function) within a class that knows which entries need to be read and passed to the module, and then records the results with the assigned entries. 

To illustrate this, we will use the simplest policy possible: a deterministic map from the observation space to the action space. For maximum clarity, we will use a simple neural network policy with the Pendulum environment we instantiated in the previous tutorial.



### Descrete action spaces

In [ ]:
import torch
from tensordict.nn import TensorDictModule
from torchrl.envs import GymEnv
from torch import nn

env = GymEnv("Pendulum-v1")

#just the shape (torch)
env_actions = env.action_spec.shape
print(f"Action space: {env_actions}")

#just the shape (number)
env_actions = env.action_spec.shape[-1]
print(f"Action space: {env_actions}")



So above we can see that the action space is a tensor of size 1. If we add [-1] we can retieve the value, not the tensor. We'll see this [-1] used a few more times.

Let's now get some more information about the action space.

In [ ]:
env.action_spec

We can see that the action_spec tells us about the shape of the actions, in this case one continunous value. It also gives us a range that he value should take (low/high). 

Let's look at the observation space.

In [ ]:
env_obs = env.observation_spec["observation"].shape[-1]
print(f"Observation space: {env_obs}")

Now that we know the observation shape (3) and the action shape (1), we can build a policy that will map observations to actions.

Let's build a simple neural network to do this.

In [ ]:
#create a model: takes observations as inputs, and outputs action
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)





If we then wrap this model in a TensorDictModule we can feed it observations and it will give us actions

In [ ]:
policy = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["action"],
)

The torchRL library has some predefined modules that do just what we did above, but for specific purposes. This can be very convinient when building our agents. 

Let's use the premade Actor module, as it works just like the code above. It takes observations and outputs actions.

In [ ]:
from torchrl.modules import Actor

#create a model: takes observations as inputs, and outputs action
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#simple to use the actor module (does the same as the TensorDict above)
policy = Actor(model)


 This is all that’s required to execute our policy! We could use a *lazy* model (i.e., LazyLinear instead of Linear) if we'd like to bypass the need to fetch the shape of the observation space, as the model will automatically determine it. But here we've explicitly told the model the shapes just for clarity.
 
 This policy is now ready to be run in the environment. To do so we can simply use a rollout, but this time specify which policy to use when the agent is determining actions.

In [ ]:
rollout = env.rollout(max_steps=10, policy=policy)
print(rollout)

We can see that it outputs a tensordict with all the typical values.

Let's take a look at the actions your policy chose to take.

In [ ]:
rollout["action"]

Our policy is still not a great policy, as it's based on a model that is randomly initalized... 

Just to explicitly show what the policy is doing, let's see how to use the policy with some observations and seeing what actions it selects. 

In [ ]:
print(policy(rollout["observation"]))


So the policy just takes the observations and uses a model to set the action to take. In this case it's a force applied to the end of the pendulm. 

We'll see that depending on the type of actions that the agent can take in the environment we'll have to build slightly different models. E.g., if the actions are left, right, nothing, then we'd have to build a categorical model. 

Let's try the steps above again this time with a different environment: one that requires discrete actions to be chosen.

### Discrete action spaces

Let's work now with the CartPole environment that takes discrete actions: either left or right movement

In [ ]:
env = GymEnv("CartPole-v1")

#Actions
env_actions = env.action_spec.shape[-1]
print(f"Action space: {env_actions}")

#Actions
env_obs = env.observation_spec["observation"].shape[-1]
print(f"Observation space: {env_obs}")

So we can see that the action space is two, but what are the values needed for these actions?

In [ ]:
env.action_spec

So we need a categorical output of size two set as a onehot encoding.

 [0,1] Right

 [1,0] Left

To do this let's create a very similar model as the one used above. Though in this case the output values will be logit values, one per action. The larger the logit value the more likely the action. 

In [ ]:
#create a model: takes observations as inputs, and outputs logits that can be used for selecting categorical actions
model = nn.Sequential(
    nn.Linear(env_obs,64),
    nn.ReLU(),
    nn.Linear(64,64),
    nn.ReLU(),
    nn.Linear(64, env_actions),
)

#go from observations to logits using the model
module = TensorDictModule(
    model,
    in_keys=["observation"],
    out_keys=["logits"]          
)


Now let's use a pre-built module, called a probabilistic actor, to choose actions based on the logits.

In [ ]:
from torchrl.modules import ProbabilisticActor
from torchrl.modules.distributions import OneHotCategorical

# go from logits to actions using the ProbabilisticActor
actor = ProbabilisticActor(
    module=module,
    in_keys=["logits"],
    out_keys=["action"],
    distribution_class=OneHotCategorical,   # converts logits values to one hot encoding: [0,1] or [1,0]
    return_log_prob=True, # return the log probabilities of the actions. this will be useful later!
)

The Probabilistic Actor will take our module and feed it observations, returing logits, and sample from a categorical distribution to give us actions in the 0,1 action space.

We'll also tell it to return the log_probability of the action taken, this will become useful later on when the agent is learning.

Let's now use this new policy when doing a rollout!

In [ ]:
#use our new actor when doing a rollout
data_rollout = env.rollout(max_steps=10, policy=actor)

#take a look
data_rollout

Let's take a look at the actions your policy made!

In [ ]:
data_rollout["action"]

Here we can see what actions our agent chose.

We can even see the logits of the different actions at each observation point.

In [ ]:
data_rollout["logits"]

We can see that our model is just choosing the action with the highest logits.

We might run into trouble here, as if the agent always chooses the highest logit, then the agent will not explore other options... let's see how we can let these kinds of agents explore.



### Explore vs Exploit

If the agent selects the highest logit it is essentially *exploiting* it's existing knowledge and choosing the best action it knows. If the agent instead selects a random action it is essentially *exploring* potential actions to see if there might be a better option out there. 

We need to know how to control this *explore vs exploit* behaviour.

Let's learn about the e-greedy option, it works well in categorical action spaces, and will let us:

> Set how often the agent choose a random action to explore. We'll call this probability of random action epsilon (eps)

> We can even set it up so that the agent starts off choosing random actions often, then as it learns better actions it can focus more on those better actions.

> This way it can start off exploring then start to exploit more as it learns more from its environment.

The torchRL library has a nice module for implementing a e-greedy strategy.

In [ ]:
from tensordict.nn import TensorDictSequential
from torchrl.modules import EGreedyModule

#create the exploration module
exploration_module = EGreedyModule(
    spec=env.action_spec,           #Action space
    eps_init= 1.0,                   #Initial exploration value: i.e., probability of choosing a random action 
    eps_end = 0.1,                   #Target exploration value
    annealing_num_steps=10000         #How many steps to go from the initial value to the target value.
    
)

#add the module to our probabilistic actor module
policy_explore = TensorDictSequential(actor, exploration_module)

So we'll start our agent off always taking a random action (eps_init) then over 10000 steps (annealing_num_steps) decrease that probability to 0.1. So at the end, after the agent has had some experience in the environment, the agent will rarely take random actions to explore, and will mostly exploit what it has already learnt.

In some environments, especialy those which require long sequences of actions to reach rewards, the agent cannot reach the those rewards if it takes random actions often. So to explore the environment more it might need to exploit it as well. 

When using the trainined agent, however, sometimes we want to turn off the explore portion of the policy completely. I.e., we are interested in using the agent in an application. It's possible to turn on and off this exploration behaviour in the policy.

Let's see how below by runing a rollout with the exploration turned on.

In [ ]:

from torchrl.envs.utils import ExplorationType, set_exploration_type

with set_exploration_type(ExplorationType.RANDOM):
    rollout_explore = env.rollout(max_steps=10, policy=policy_explore)

rollout_explore

Let's take a look at the actions.

In [ ]:
rollout_explore["action"]

If we compare those actions take to the logits we can see if the agent is always taking the best action, or randomly taking actions as well.

In [ ]:
rollout_explore["logits"]

**Things to try**

* Try to load in the [mountain car environment](https://gymnasium.farama.org/environments/classic_control/mountain_car/) and build a policy for it. 

* Can you get the e-greedy exploration added to the policy?

* Can you visualize your environment?


### Next Steps

Now we know how to setup environments, build policies, and use these policies to guide the actions of our agents.

We've also seen how to setup continuous and discrete action spaces.

But our policies are really bad at the moment... in the next tutorial let's look at how we can allow these policies to get better (i.e., how do we get our agents to learn?!)


In the meantime:
* Check out the official tutorial [here](https://docs.pytorch.org/rl/stable/tutorials/getting-started-1.html).
* Read up more about other [modules](https://docs.pytorch.org/rl/main/reference/modules.html).
